## PREPROCESSING

In [9]:
%pip install ucimlrepo iterative-stratification

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

Fetch data

In [11]:
from ucimlrepo import fetch_ucirepo
mi = fetch_ucirepo(id=579)


Define feature and target sets

In [12]:
X_raw = mi.data.features.copy()
y = mi.data.targets.copy()

print("Feature shape:", X_raw.shape)
print("Target shape:", y.shape)

Feature shape: (1700, 111)
Target shape: (1700, 12)


Drop Features

In [13]:
drop_columns = [
    "R_AB_3_n",
    "NA_R_3_n",
    "NOT_NA_3_n",
    
]

X_48h = X_raw.drop(columns=drop_columns)

X_drop_1 = X_48h.drop(columns=["KFK_BLOOD", "IBS_NASL"])

print("Missing Value Feature Drop input shape:", X_drop_1.shape)



Missing Value Feature Drop input shape: (1700, 106)


Create Binary Indicator Column "EMR_RES_BLOOD" for D_AD_KBRIG and S_AD_KBRIG

In [14]:
X_indicator = X_drop_1.copy()

X_indicator["EMR_RES_BLOOD"] = (
    X_indicator["D_AD_KBRIG"].notna().astype(int)
)


print("OG 48-hour features:", X_48h.shape)
print("After removing high missing KFK_BLOOD and IBS_NASL:", X_drop_1.shape)
print("After adding Binary indicator for emergency response:", X_indicator.shape)


OG 48-hour features: (1700, 108)
After removing high missing KFK_BLOOD and IBS_NASL: (1700, 106)
After adding Binary indicator for emergency response: (1700, 107)


Define feature sets by data type

In [15]:
# Numerical features
numerical_measurement_column_names_keep = [
    "K_BLOOD", "NA_BLOOD", "ALT_BLOOD", "AST_BLOOD",
    "L_BLOOD", "ROE", "AGE", "S_AD_ORIT", "D_AD_ORIT",
    "S_AD_KBRIG", "D_AD_KBRIG"
]

# Ordinal features
ordinal_column_names_keep = [
    "INF_ANAM", "STENOK_AN", "FK_STENOK", "GB", "DLIT_AG",
    "ant_im", "lat_im", "inf_im", "post_im", "TIME_B_S",
    "R_AB_1_n", "R_AB_2_n", "NOT_NA_1_n", "IBS_POST",
    "NA_R_1_n", "NA_R_2_n", "NOT_NA_2_n"
]

# Partially ordered feature
partially_ordered_column_names_keep = [
    "ZSN_A"
]

# Binary features
binary_column_names_keep = [
    "SEX", "SIM_GIPERT", "nr_11", "nr_01", "nr_02", "nr_03",
    "nr_04", "nr_07", "nr_08", "np_01", "np_04", "np_05",
    "np_07", "np_08", "np_09", "np_10", "endocr_01",
    "endocr_02", "endocr_03", "zab_leg_01", "zab_leg_02",
    "zab_leg_03", "zab_leg_04", "zab_leg_06", "O_L_POST",
    "K_SH_POST", "MP_TP_POST", "SVT_POST", "GT_POST",
    "FIB_G_POST", "IM_PG_P", "ritm_ecg_p_01", "ritm_ecg_p_02",
    "ritm_ecg_p_04", "ritm_ecg_p_06", "ritm_ecg_p_07",
    "ritm_ecg_p_08", "n_r_ecg_p_01", "n_r_ecg_p_02",
    "n_r_ecg_p_03", "n_r_ecg_p_04", "n_r_ecg_p_05",
    "n_r_ecg_p_06", "n_r_ecg_p_08", "n_r_ecg_p_09",
    "n_r_ecg_p_10", "n_p_ecg_p_01", "n_p_ecg_p_03",
    "n_p_ecg_p_04", "n_p_ecg_p_05", "n_p_ecg_p_06",
    "n_p_ecg_p_07", "n_p_ecg_p_08", "n_p_ecg_p_09",
    "n_p_ecg_p_10", "n_p_ecg_p_11", "n_p_ecg_p_12",
    "fibr_ter_01", "fibr_ter_02", "fibr_ter_03",
    "fibr_ter_05", "fibr_ter_06", "fibr_ter_07",
    "fibr_ter_08", "GIPO_K", "GIPER_NA", "NA_KB",
    "NOT_NA_KB", "LID_KB", "NITR_S", "LID_S_n",
    "B_BLOK_S_n", "ANT_CA_S_n", "GEPAR_S_n", "ASP_S_n",
    "TIKL_S_n", "TRENT_S_n", "EMR_RES_BLOOD"
]

# Features used for mode imputation
other_feature_column_names_keep = (
    binary_column_names_keep
    + ordinal_column_names_keep
    + partially_ordered_column_names_keep
)

print("Other features (binary + ordinal + partially ordered)")
print("Count:", len(other_feature_column_names_keep))


print("Features in sets features:", len(ordinal_column_names_keep + binary_column_names_keep + partially_ordered_column_names_keep + numerical_measurement_column_names_keep))
print("Total features:", X_indicator.shape)

Other features (binary + ordinal + partially ordered)
Count: 96
Features in sets features: 107
Total features: (1700, 107)


Grab the actual X and y datsets

In [16]:
# Features(inputs)
X = X_indicator.copy()

# Targets(outputs)
y = mi.data.targets.copy()

print("Features shape:", X.shape)
print("Targets shape:", y.shape)


Features shape: (1700, 107)
Targets shape: (1700, 12)


One Hot encode LET_IS 8 class target column to prep for Iterative Stratification

In [17]:
#grab LET_IS column
let_is = y["LET_IS"]

#one hot encodeing 
let_is_encode = pd.get_dummies(let_is, prefix="LET_IS", dtype=int)

#Drop LET_IS from the targets
dropped_let_is = y.drop(columns=["LET_IS"])

#Combine the encoded LET_IS columns with the bianry ones
new_y = pd.concat([dropped_let_is, let_is_encode], axis=1)

#should have 19 columns (11 binary + 8 new ones from LET_IS)
print(new_y.shape)

(1700, 19)


Iterative Stratification

In [18]:
#split the train set first
first_split = MultilabelStratifiedShuffleSplit(n_splits=1,test_size=0.30,random_state=42)

# do the split and create indices
train_split = first_split.split(X, new_y)

# Grab the indices
training_indices, to_split_indices = next(train_split)

print("Training indices:", len(training_indices))
print("Temporary indices:", len(to_split_indices))

#split the to_split set into validation and test sets
second_split = MultilabelStratifiedShuffleSplit(n_splits=1,test_size=0.50,random_state=42)

#grab the rows corresponsing to the indices in to_split_set
to_split_X = X.iloc[to_split_indices]
to_split_y = new_y.iloc[to_split_indices]

#split and create indices
validation_test_split = second_split.split(to_split_X, to_split_y)

#grab the indices - these are relative to the to_split set so we have to map them to the actual dataset
validation_indices_1, test_indices_1 = next(validation_test_split)

#convert indices so they are relative to actual dataset
validation_indices = to_split_indices[validation_indices_1]
test_indices = to_split_indices[test_indices_1]


print("Training:", len(training_indices))
print("Validation:", len(validation_indices))
print("Test:", len(test_indices))

Training indices: 1190
Temporary indices: 510
Training: 1190
Validation: 255
Test: 255


Grab those index rows(using indices determined by Iterative Stratification) from our original datset (X and y) without the one hot encoded LET_IS

In [19]:
#get X_train set
X_train_rows = X.iloc[training_indices]
X_train = X_train_rows.copy()

#get y_train set
y_train_rows = y.iloc[training_indices]
y_train = y_train_rows.copy()

#get X_validation set
X_validation_rows = X.iloc[validation_indices]
X_validation = X_validation_rows.copy()

#get y_validation set
y_validation_rows = y.iloc[validation_indices]
y_validation = y_validation_rows.copy()

#get X_test set
X_test_rows = X.iloc[test_indices]
X_test = X_test_rows.copy()

#get y_test set
y_test_rows = y.iloc[test_indices]
y_test = y_test_rows.copy()

print(X_train.shape)
print(y_train.shape)

print(X_validation.shape)
print(y_validation.shape)

print(X_test.shape)
print(y_test.shape)

(1190, 107)
(1190, 12)
(255, 107)
(255, 12)
(255, 107)
(255, 12)


Median Imputer for Numerical Columns

In [20]:
# Create the median imputer
median_imputer = SimpleImputer(strategy="median")

# Grab the numerical features from the training set
X_train_numerical = X_train[numerical_measurement_column_names_keep]

#calculate the median based on training data only
median_imputer.fit(X_train_numerical)

#impute the median values for the numerical columns in training set
X_train[numerical_measurement_column_names_keep] = median_imputer.transform(X_train[numerical_measurement_column_names_keep])

#use the same median and apply to validation and test set(once again we use the calculated values from traning)
X_validation[numerical_measurement_column_names_keep] = median_imputer.transform(X_validation[numerical_measurement_column_names_keep])
X_test[numerical_measurement_column_names_keep] = median_imputer.transform(X_test[numerical_measurement_column_names_keep])




Mode Imputer for Binary + Partially Ordered + Ordinal columns

In [21]:
#create the mode imputer
mode_imputer = SimpleImputer(strategy="most_frequent")

#Grab the categorical and binary features from training set
X_train_other = X_train[other_feature_column_names_keep]

#calculate the mode based on training data only
mode_imputer.fit(X_train_other)

#impute the mode values for the columns in training set
X_train[other_feature_column_names_keep] = mode_imputer.transform(X_train[other_feature_column_names_keep])

#use the same mode and apply to validation and test set(once again we use the calculated values from traning)
X_validation[other_feature_column_names_keep] = mode_imputer.transform(X_validation[other_feature_column_names_keep])
X_test[other_feature_column_names_keep] = mode_imputer.transform(X_test[other_feature_column_names_keep])

Scale the numerical values using standard scaler

In [22]:
scaler = StandardScaler()

#detrmine how to scale using training data only
scaler.fit(X_train[numerical_measurement_column_names_keep])

#scale the training set
X_train[numerical_measurement_column_names_keep] = scaler.transform(X_train[numerical_measurement_column_names_keep])

#scale validation set
X_validation[numerical_measurement_column_names_keep] = scaler.transform(X_validation[numerical_measurement_column_names_keep])

#scale test set
X_test[numerical_measurement_column_names_keep] = scaler.transform(X_test[numerical_measurement_column_names_keep])


Scale Ordinal + ZSN_A column(will treat like ordinal for now - might make other preprocessing decisions in the future)

In [23]:
ordinal_column_names_keep = ordinal_column_names_keep + ["ZSN_A"]

mix_max_scaler = MinMaxScaler()

#detrmine how to scale using training data only
mix_max_scaler.fit(X_train[ordinal_column_names_keep])

# Scale training set
X_train[ordinal_column_names_keep] = mix_max_scaler.transform(X_train[ordinal_column_names_keep])

# Scale validation set using only the training calc values
X_validation[ordinal_column_names_keep] = mix_max_scaler.transform(X_validation[ordinal_column_names_keep])

# Scale test set using only training calc values
X_test[ordinal_column_names_keep] = mix_max_scaler.transform(X_test[ordinal_column_names_keep])

In [24]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
import pandas as pd

baseline_results = []
baseline_models = {}

# Train one majority-class classifier per target
for target in y_train.columns:

    baseline = DummyClassifier(strategy="most_frequent")

    # Fit using training data only
    baseline.fit(X_train, y_train[target])

    # Predict on validation data
    y_pred = baseline.predict(X_validation)

    # Store model
    baseline_models[target] = baseline

    # Calculate evaluation metrics
    baseline_results.append({
        "Target": target,
        "Majority Class": y_train[target].mode()[0],
        "Accuracy": accuracy_score(y_validation[target], y_pred),
        "Macro Precision": precision_score(
            y_validation[target], y_pred,
            average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_validation[target], y_pred,
            average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_validation[target], y_pred,
            average="macro", zero_division=0
        )
    })

baseline_df = pd.DataFrame(baseline_results)

print(baseline_df.round(4).to_string(index=False))


    Target  Majority Class  Accuracy  Macro Precision  Macro Recall  Macro F1
FIBR_PREDS               0    0.8980           0.4490         0.500    0.4731
 PREDS_TAH               0    0.9882           0.4941         0.500    0.4970
 JELUD_TAH               0    0.9725           0.4863         0.500    0.4930
FIBR_JELUD               0    0.9569           0.4784         0.500    0.4890
  A_V_BLOK               0    0.9686           0.4843         0.500    0.4920
 OTEK_LANC               0    0.9059           0.4529         0.500    0.4753
    RAZRIV               0    0.9686           0.4843         0.500    0.4920
  DRESSLER               0    0.9569           0.4784         0.500    0.4890
       ZSN               0    0.7686           0.3843         0.500    0.4346
    REC_IM               0    0.9059           0.4529         0.500    0.4753
 P_IM_STEN               0    0.9137           0.4569         0.500    0.4775
    LET_IS               0    0.8431           0.1054         0.

Preprocessing is done.

For model training, please use:

- X_train, y_train
- X_validation, y_validation
- X_test, y_test

**Important:** Please restart the kernel and run all preprocessing cells in order to get accurately preprocessed datasets.